# Optional HDF5 and comparison matrix preparation
For large collections of directed pairs, including all-against-all. This is downstream processing, not a compulsory pipeline stage. The retained matching function uses full containment and normalizes interval boundaries with min/max. It stops at the first search gap with any match. This differs deliberately from the pairwise notebook. Pair/source matrices and target BED indices still need RAM. A failed build remains as `.building.h5` for inspection; an existing final HDF5 is replaced only after success.

In [ ]:
from pathlib import Path
import json
import sys
import csv
from collections import Counter
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "bin" / "annotation_io.py").is_file():
    raise RuntimeError("Start Jupyter from the repository or its notebooks directory")
sys.path.insert(0, str(ROOT / "bin"))
from annotation_io import iter_bed

In [ ]:
GENOMES_TSV = ROOT / "examples/genomes.tsv"
PAIRS_TSV = ROOT / "examples/pairs.tsv"
RESULTS_DIR = ROOT / "results"
OUTPUT_DIR = RESULTS_DIR / "analysis"
SEARCH_GAPS = [50, 55, 65, 75, 85, 100]
VALID_TRANSFER_STATUSES = {
    "occupied_expected_length", "occupied_different_length",
    "local_indel_or_rearranged", "empty_with_TSD", "empty_without_TSD",
}
CHUNKSIZE = 250_000

WORK_H5 = OUTPUT_DIR / "comparison.h5"
REBUILD = False

In [ ]:
def manifest_path(manifest, value):
    path = Path(value)
    return path if path.is_absolute() else manifest.parent / path

GENOMES = pd.read_csv(GENOMES_TSV, sep="\t", dtype=str, keep_default_na=False).set_index("genome_id", verify_integrity=True)
if not GENOMES.index.to_series().str.fullmatch(r"[A-Za-z][A-Za-z0-9_]*").all():
    raise ValueError("Invalid genome_id")
PAIRS = pd.read_csv(PAIRS_TSV, sep="\t", dtype=str, keep_default_na=False)
if PAIRS.duplicated(["source", "target"]).any():
    raise ValueError("Duplicate directed pair")
if not set(PAIRS.source).union(PAIRS.target).issubset(GENOMES.index):
    raise ValueError("Pair manifest refers to an unknown genome")

def statuses_path(pair):
    supplied = getattr(pair, "statuses", "")
    return manifest_path(PAIRS_TSV, supplied) if supplied else RESULTS_DIR / "pairs" / f"{pair.source}__to__{pair.target}" / "statuses.tsv"

def bed_input(genome_id):
    row = GENOMES.loc[genome_id]
    published = RESULTS_DIR / "genomes" / genome_id / "repeats.bed"
    if published.is_file():
        return published, "bed6", {}
    if row.get("annotation_format", "") in {"bed6", "custom5"} and row.get("annotation", ""):
        chrom_map = {}
        if row.get("chrom_map", ""):
            chrom_map = json.loads(manifest_path(GENOMES_TSV, row.chrom_map).read_text())
        return manifest_path(GENOMES_TSV, row.annotation), row.annotation_format, chrom_map
    raise FileNotFoundError(f"No BED annotation for {genome_id}; export repeats.bed first")

def read_repeats(genome_id):
    path, format_name, chrom_map = bed_input(genome_id)
    return pd.DataFrame(
        [(chrom_map.get(x.chrom, x.chrom), x.start0, x.end0, x.strand, x.rep_name)
         for x in iter_bed(path, format_name)],
        columns=["chr", "start", "end", "strand", "rep_name"],
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_repeatmasker_index(genome_id):
    rep = read_repeats(genome_id)
    rep = rep.sort_values(["chr", "start", "end"], kind="mergesort").reset_index(drop=True)
    rep["repeat_index"] = rep.index.astype("int64")
    return {
        str(chrom): {
            "starts": grp.start.to_numpy(dtype=np.int64),
            "ends": grp.end.to_numpy(dtype=np.int64),
            "indices": grp.repeat_index.to_numpy(dtype=np.int64),
            "names": grp.rep_name.astype(str).to_numpy(),
        }
        for chrom, grp in rep.groupby("chr", sort=False)
    }

In [ ]:
def repeat_name_matches_source(
    source_repeat_name,
    target_repeat_names,
):
    source_repeat_name = str(source_repeat_name)

    return any(
        source_repeat_name == str(target_repeat_name)
        for target_repeat_name in target_repeat_names
    )

In [ ]:
def classify_locus_against_repeatmasker(
    row,
    rep_by_chr,
):
    if row.status not in VALID_TRANSFER_STATUSES:
        return "not_transferred"

    chrom_index = rep_by_chr.get(str(row.target_contig))

    if chrom_index is None:
        return "transferred_no_repeat"

    starts = chrom_index["starts"]
    ends = chrom_index["ends"]
    names = chrom_index["names"]

    matched_names = []

    for gap in SEARCH_GAPS:
        if row.target_strand == "+":
            raw_left = row.left_target_end - gap
            raw_right = row.right_target_start + gap

        elif row.target_strand == "-":
            raw_left = row.right_target_end - gap
            raw_right = row.left_target_start + gap

        else:
            return "transferred_no_repeat"

        interval_left = int(min(raw_left, raw_right))
        interval_right = int(max(raw_left, raw_right))

        first_candidate = np.searchsorted(
            starts,
            interval_left,
            side="left",
        )

        last_candidate = np.searchsorted(
            starts,
            interval_right,
            side="right",
        )

        if last_candidate <= first_candidate:
            continue

        candidate_ends = ends[first_candidate:last_candidate]

        fully_contained = candidate_ends <= interval_right

        if not fully_contained.any():
            continue

        matched_names = (
            names[first_candidate:last_candidate][fully_contained]
            .tolist()
        )

        if matched_names:
            break

    n_repeats = len(matched_names)

    if n_repeats == 0:
        return "transferred_no_repeat"

    name_matches = repeat_name_matches_source(
        source_repeat_name=row.repeat_ids_inside,
        target_repeat_names=matched_names,
    )

    if n_repeats == 1:
        if name_matches:
            return "transferred_one_repeat_same_name"

        return "transferred_one_repeat_different_name"

    if name_matches:
        return "transferred_multiple_repeats_with_source_name"

    return "transferred_multiple_repeats_without_source_name"

In [ ]:
TRANSFER_KEY = "transfers"
MATRIX_KEY = "repeat_annotation_status_matrix"
META_COLUMNS = ["source_tag", "source_genome_id", "group_id", "ref_chrom", "ref_start", "ref_end", "repeat_len", "repeat_ids_inside"]
INPUT_COLUMNS = ["group_id", "ref_chrom", "ref_start", "ref_end", "repeat_len", "repeat_ids_inside", "status", "target_contig", "target_strand", "left_target_start", "left_target_end", "right_target_start", "right_target_end"]
GENOME_TAGS = GENOMES.index.tolist()
if set(GENOME_TAGS).intersection(META_COLUMNS):
    raise ValueError("Genome IDs conflict with matrix metadata columns")
if WORK_H5.exists() and not REBUILD:
    raise FileExistsError("HDF5 already exists; use the statistics notebook or explicitly set REBUILD=True")
BUILD_H5 = WORK_H5.with_suffix(".building.h5")
if BUILD_H5.exists():
    raise FileExistsError("Incomplete HDF5 build exists; inspect it before removing it manually")
WORK_H5.parent.mkdir(parents=True, exist_ok=True)
for pair in PAIRS.itertuples(index=False):
    if not statuses_path(pair).is_file():
        raise FileNotFoundError(statuses_path(pair))
    bed_input(pair.target)

string_widths = {"group_id":128,"ref_chrom":128,"repeat_ids_inside":256,"status":64,"target_contig":128,"target_strand":1,
                 "source_tag":max(32,max(map(len,GENOME_TAGS))),"source_genome_id":max(32,max(map(len,GENOME_TAGS))),
                 "target_tag":max(32,max(map(len,GENOME_TAGS)))}
with pd.HDFStore(BUILD_H5, mode="w") as store:
    for pair in tqdm(list(PAIRS.itertuples(index=False)), desc="Import TSV"):
        seen = set()
        for chunk in pd.read_csv(statuses_path(pair), sep="\t", usecols=INPUT_COLUMNS,
                                 chunksize=CHUNKSIZE, dtype=str, keep_default_na=False):
            ids = chunk.group_id.tolist()
            if len(set(ids)) != len(ids) or seen.intersection(ids):
                raise ValueError("Duplicate group_id; no records have been silently discarded")
            seen.update(ids)
            for col in ["ref_start","ref_end","repeat_len"]:
                chunk[col] = pd.to_numeric(chunk[col], errors="raise").astype("int64")
            for col in ["left_target_start","left_target_end","right_target_start","right_target_end"]:
                chunk[col] = pd.to_numeric(chunk[col].replace("", np.nan), errors="raise").astype("float64")
            chunk["source_tag"] = pair.source
            chunk["source_genome_id"] = pair.source
            chunk["target_tag"] = pair.target
            store.append(TRANSFER_KEY, chunk, format="table", data_columns=["source_tag","target_tag","group_id"],
                         min_itemsize=string_widths, index=False)
    if f"/{TRANSFER_KEY}" not in store:
        raise ValueError("No status records were imported")
    store.create_table_index(TRANSFER_KEY, columns=["source_tag","target_tag"], optlevel=9, kind="full")
    store.get_storer(TRANSFER_KEY).attrs.provenance = json.dumps({
        "genomes_manifest":GENOMES_TSV.read_text(), "pairs_manifest":PAIRS_TSV.read_text(),
        "results_dir":str(RESULTS_DIR.resolve()), "search_gaps":SEARCH_GAPS,
        "valid_transfer_statuses":sorted(VALID_TRANSFER_STATUSES),
        "boundary_rule":"min_max_normalized", "input_columns":INPUT_COLUMNS,
    })

In [ ]:
matrix_widths = {key:value for key,value in string_widths.items() if key in META_COLUMNS}
matrix_widths.update({tag:64 for tag in GENOME_TAGS})
with pd.HDFStore(BUILD_H5, mode="a") as store:
    for source, source_pairs in tqdm(PAIRS.groupby("source", sort=False), desc="Source matrices"):
        matrix = None
        metadata = None
        for pair in source_pairs.itertuples(index=False):
            rep_by_chr = load_repeatmasker_index(pair.target)
            parts = []
            where = f"(source_tag == {source!r}) & (target_tag == {pair.target!r})"
            for chunk in store.select(TRANSFER_KEY, where=where, iterator=True, chunksize=20_000):
                compact = chunk[META_COLUMNS].copy()
                compact["annotation_status"] = [classify_locus_against_repeatmasker(row, rep_by_chr)
                                                 for row in chunk.itertuples(index=False)]
                parts.append(compact)
            if not parts:
                continue
            pair_result = pd.concat(parts, ignore_index=True).set_index("group_id", verify_integrity=True)
            pair_meta = pair_result.drop(columns="annotation_status")
            if metadata is None:
                metadata = pair_meta
                matrix = pair_result[["annotation_status"]].rename(columns={"annotation_status":pair.target})
            else:
                shared = metadata.index.intersection(pair_meta.index)
                if not metadata.loc[shared].equals(pair_meta.loc[shared]):
                    raise ValueError(f"Conflicting locus metadata for source {source}")
                # Preserve metadata for loci present in only some directed comparisons.
                metadata = metadata.combine_first(pair_meta)
                matrix = matrix.join(pair_result.annotation_status.rename(pair.target), how="outer", validate="one_to_one")
            del rep_by_chr, parts, pair_result
        if metadata is None:
            continue
        matrix[source] = "self_no_transfer"
        for tag in GENOME_TAGS:
            if tag not in matrix:
                matrix[tag] = "not_compared"
        matrix[GENOME_TAGS] = matrix[GENOME_TAGS].fillna("not_compared")
        result = metadata.join(matrix).reset_index()[META_COLUMNS + GENOME_TAGS]
        for col in ["ref_start","ref_end","repeat_len"]:
            result[col] = result[col].astype("int64")
        result = result.sort_values(["ref_chrom","ref_start","ref_end","group_id"], kind="mergesort")
        store.append(MATRIX_KEY, result, format="table", data_columns=["source_tag","group_id"],
                     min_itemsize=matrix_widths, index=False)
    if f"/{MATRIX_KEY}" not in store:
        raise ValueError("No matrix rows were generated")
    store.create_table_index(MATRIX_KEY, columns=["source_tag","group_id"], optlevel=9, kind="full")
    store.get_storer(MATRIX_KEY).attrs.genome_tags = GENOME_TAGS
BUILD_H5.replace(WORK_H5)
print(WORK_H5)